In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import os

# 1. Configuration
# Names of your three output files
output_files = ['Rag_Output.csv', 'inference_output.csv', 'fine_tune_output.csv']
reference_file = 'Dataset_clean_Answers.csv'

# Load model (paraphrase-multilingual-MiniLM-L12-v2)
# This model is excellent for German/English cross-lingual or multilingual similarity
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# List to store final summary results
summary_results = []

# 2. Load Reference Data
# Using sep=';' and latin-1 encoding based on your specific file structure
df_ref = pd.read_csv(reference_file,
                     sep=';',
                     encoding='latin-1',
                     usecols=[0, 1],
                     names=['id', 'correct_answer'],
                     header=0)

print(f"Reference data loaded successfully ({len(df_ref)} rows).\n")

# 3. Processing Loop
for file_name in output_files:
    if not os.path.exists(file_name):
        print(f"File not found: {file_name} - skipping.")
        continue

    print(f"Analyzing {file_name}...")

    # Load Output file
    df_output = pd.read_csv(file_name)

    # Merge data (matching by ID)
    df_merged = pd.merge(df_output[['id', 'answer']], df_ref, on='id', how='inner')

    if df_merged.empty:
        print(f"No matching IDs found in {file_name}.")
        continue

    # Vectorize answers (Embeddings)
    emb_output = model.encode(df_merged['answer'].fillna("").astype(str).tolist(), convert_to_tensor=True)
    emb_ref = model.encode(df_merged['correct_answer'].fillna("").astype(str).tolist(), convert_to_tensor=True)

    # Calculate Cosine Similarity
    # We take the diagonal of the matrix to compare pairs 1:1
    cosine_scores = util.cos_sim(emb_output, emb_ref)
    similarities = [cosine_scores[i][i].item() for i in range(len(df_merged))]

    # A) Save detailed CSV for this specific file
    df_detail = df_merged.copy()
    df_detail['similarity_score'] = similarities

    detail_name = f"Similarity_Detail_{file_name}"
    df_detail.to_csv(detail_name, index=False)

    # B) Calculate average for the main summary
    avg_score = df_detail['similarity_score'].mean()
    summary_results.append({
        'Output_File': file_name,
        'Avg_BERT_Score': round(avg_score, 4),
        'Total_Answers': len(df_detail)
    })

    print(f"Successfully created: {detail_name} (Mean Score: {avg_score:.4f})")

# 4. Create Main Results CSV
df_main = pd.DataFrame(summary_results)
df_main.to_csv('Main_Comparison_Results.csv', index=False)

print("\n" + "="*40)
print("EVALUATION COMPLETE")
print("Summary saved in: 'Main_Comparison_Results.csv'")
print("="*40)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Reference data loaded successfully (645 rows).

Analyzing Rag_Output.csv...
Successfully created: Similarity_Detail_Rag_Output.csv (Mean Score: 0.5569)
Analyzing inference_output.csv...
Successfully created: Similarity_Detail_inference_output.csv (Mean Score: 0.6822)
Analyzing fine_tune_output.csv...
Successfully created: Similarity_Detail_fine_tune_output.csv (Mean Score: 0.4761)

EVALUATION COMPLETE
Summary saved in: 'Main_Comparison_Results.csv'
